# V1

In [24]:
def print_polynomial(f):
    result = ""
    for i in range(len(f)):
        coefficient = str(f[i])
        if coefficient == "0":
            continue
        if i == 0:
            result += f"{coefficient} + "
        elif i == 1:
            if coefficient == "1":
                coefficient = ""
            result += f"{coefficient}x + "
        else:
            if coefficient == "1":
                coefficient = ""
            result += f"{coefficient}x^{i} + "
    print(result.rstrip(" + ")) 

def print_vector(a):
    print()
    for polynomial in a:
        print_polynomial(polynomial)
    print()

In [10]:
def add_polynomials(f, g, modulus):
    result_length = max(len(f), len(g))
    result = [0] * result_length
    for i in range(result_length):
        coefficient_f = f[i] if i < len(f) else 0
        coefficient_g = g[i] if i < len(g) else 0
        result[i] = (coefficient_f + coefficient_g) % modulus
    return result

def subtract_polynomials(f, g, modulus):
    result_length = max(len(f), len(g))
    result = [0] * result_length
    for i in range(result_length):
        coefficient_f = f[i] if i < len(f) else 0
        coefficient_g = g[i] if i < len(g) else 0
        result[i] = (coefficient_f - coefficient_g) % modulus
    return result

def multiply_polynomials(f, g, modulus):
    result_length = len(f) + len(g) - 1
    result = [0] * result_length
    for i in range(len(f)):
        for j in range(len(g)):
            result[i+j] = (result[i+j] + f[i] * g[j]) % modulus
    return result

In [27]:
f = [5, 0, 4, 3]
g = [6, 3, 2]

print_polynomial(f)
print_polynomial(g)

h = add_polynomials(f, g, 7)
print_polynomial(h)

i = subtract_polynomials(f, g, 7)
print_polynomial(i)

j = multiply_polynomials(f, g, 7)
print_polynomial(j)

5 + 4x^2 + 3x^3
6 + 3x + 2x^2
4 + 3x + 6x^2 + 3x^3
6 + 4x + 2x^2 + 3x^3
2 + x + 6x^2 + 2x^3 + 3x^4 + 6x^5


In [26]:
def ring_multiply_polynomials(f, g, modulus, n):
    result = multiply_polynomials(f, g, modulus)
    if len(result) > n:
        for i in range(n, len(result)):
            result[i%n] = (result[i%n] - result[i]) % modulus
        result = result[:n]
    return result

In [28]:
f = [32,0,17,22]
g = [11,7,19,1]
h = ring_multiply_polynomials(f, g, 41, 4)
print_polynomial(h)

39 + 35x + 35x^2 + 24x^3


In [29]:
f = [23,0,11,7]
g = [40,5,16,0]
print(add_polynomials(f, g, 41))
print(subtract_polynomials(f, g, 41))
print(ring_multiply_polynomials(f, g, 41, 4))


[22, 5, 27, 7]
[24, 36, 36, 7]
[12, 3, 29, 7]


In [30]:
def add_vectors(a, b, modulus, k):
    result = []
    for i in range(k):
        result.append(add_polynomials(a[i], b[i], modulus))
    return result

def subtract_vectors(a, b, modulus, k):
    result = []
    for i in range(k):
        result.append(subtract_polynomials(a[i], b[i], modulus))
    return result

def multiply_vectors(a, b, modulus, k, n):
    result = []
    for i in range(k):
        multiplied = ring_multiply_polynomials(a[i], b[i], modulus, n)
        result = add_polynomials(result, multiplied, modulus)
    return result

In [31]:
a = [[93,51,34,54],[27,87,81,6],[112,15,46,122]]
b = [[40,78,1,119],[11,31,57,90],[108,72,47,14]]

print_vector(add_vectors(a, b, 137, 3))
print_vector(subtract_vectors(a, b, 137, 3))

print_polynomial(multiply_vectors(a, b, 137, 3, 4))



133 + 129x + 35x^2 + 36x^3
38 + 118x + x^2 + 96x^3
83 + 87x + 93x^2 + 136x^3


53 + 110x + 33x^2 + 72x^3
16 + 56x + 24x^2 + 53x^3
4 + 80x + 136x^2 + 108x^3

93 + 59x + 44x^2 + 132x^3


In [13]:
def mod_s(r, q):
    r = r % q

    if q % 2 == 0:
        if r <= q // 2:
            result = r
        else:
            result = r - q
    else:
        if r <= (q - 1) // 2:
            result = r
        else:
            result = r - q

    return result


In [33]:
print(mod_s(9+15, 17))
print(mod_s(9-15, 17))
print(mod_s(9*15, 17))
print()
print(mod_s(9+15, 18))
print(mod_s(9-15, 18))
print(mod_s(9*15, 18))

7
-6
-1

6
-6
9


In [34]:
def integer_size(r, q):
    return abs(mod_s(r, q))

def polynomial_size(f, q):
    return max([integer_size(coefficient, q) for coefficient in f])

def vector_size(a, q):
    return max([polynomial_size(polynomial, q) for polynomial in a])

In [35]:
print(integer_size(7, 19))
print(integer_size(18, 19))
print(polynomial_size([1, 12, 0, 3, 0, 18], 19))

7
1
7


# V2

## V2a - simplifies scheme

In [36]:
def round_q(x, q):
    x_prime = mod_s(x, q)
    if -int(q/4) <= x_prime <= int(q/4):
        return 0
    else:
        return 1
    
def round_polynomial(f, q):
    return [round_q(coefficient, q) for coefficient in f]

In [37]:
print_polynomial(round_polynomial([3000, 1500, 2010, 37], 3329))

x + x^2


In [55]:
def multiply_matrix_by_vector(A, b, modulus, k, n):
    result = [[] for _ in range(k)]

    for row_A in range(k):
        result[row_A] = multiply_vectors(A[row_A], b, modulus, k, n)

    return result

In [56]:
q = 137
n = 4
k = 2
eta1 = 2
eta2 = 2

A = [[[21,57,78,43],[126,122,19,125]],
     [[111,9,63,33],[105,61,71,64]]]

s = [[1,2,-1,2],[0,-1,0,2]]
e = [[1,0,-1,1],[0,-1,1,0]]

As = multiply_matrix_by_vector(A, s, q, k, n)
t = add_vectors(As, e, q, k)

print_vector(t)


55 + 96x + 123x^2 + 7x^3
32 + 27x + 127x^2 + 100x^3



In [ ]:
def bitstring_to_polynomial(m, n):
    f = [0] * n
    for i in range(n):
        if m[i] == "1":
            f[i] = 1
    return f

def polynomial_to_bitstring(f, n):
    m = ""
    for i in range(n):
        if f[i] == 1:
            m += "1"
        else:
            m += "0"
    return m

In [59]:
def multiply_polynomial_by_scalar(f, scalar, modulus):
    return [(coefficient * scalar) % modulus for coefficient in f]

In [69]:
def matrix_transpose(A, k, n):
    result = [[0] * k for _ in range(n)]
    for i in range(k):
        for j in range(n):
            result[j][i] = A[i][j]
    return result

In [ ]:
plaintext = "0111"
m = bitstring_to_polynomial(plaintext, n)

r = [[-2,2,1,-1],[-1,1,1,0]]

e1 = [[1,0,-2,1],[-1,2,-2,1]]

e2 = [2,2,-1,1]

AT = matrix_transpose(A, k, k)

ATr = multiply_matrix_by_vector(AT, r, q, k, n)

u = add_vectors(ATr, e1, q, k)

print_vector(u)

tTr = multiply_vectors(t, r, q, k, n)

half_qm = multiply_polynomial_by_scalar(m, (q+1)//2, q)

tTr_e2 = add_polynomials(tTr, e2, q)

v = add_polynomials(tTr_e2, half_qm, q)

print_polynomial(v)


56 + 32x + 77x^2 + 9x^3
45 + 21x + 2x^2 + 127x^3

3 + 10x + 8x^2 + 123x^3


In [ ]:
sTu = multiply_vectors(s, u, q, k, n)
d = subtract_polynomials(v, sTu, q)
print_polynomial(d)
print_polynomial(round_polynomial(d, q))

plaintext_recovered = polynomial_to_bitstring(round_polynomial(d, q), n)
print(plaintext_recovered)

4 + 60x + 79x^2 + 66x^3
x + x^2 + x^3
0111


### Example of fully functional simplified Kyber-PKE scheme

In [171]:
from random import randint
from dataclasses import dataclass

In [172]:
@dataclass(frozen=True)
class KyberParameters:
    q: int
    n: int
    k: int
    eta1: int
    eta2: int

In [173]:
class MathOperations:
    @staticmethod
    def add_polynomials(f, g, modulus):
        result_length = max(len(f), len(g))
        result = [0] * result_length
        for i in range(result_length):
            coefficient_f = f[i] if i < len(f) else 0
            coefficient_g = g[i] if i < len(g) else 0
            result[i] = (coefficient_f + coefficient_g) % modulus
        return result

    @staticmethod
    def subtract_polynomials(f, g, modulus):
        result_length = max(len(f), len(g))
        result = [0] * result_length
        for i in range(result_length):
            coefficient_f = f[i] if i < len(f) else 0
            coefficient_g = g[i] if i < len(g) else 0
            result[i] = (coefficient_f - coefficient_g) % modulus
        return result
    
    @staticmethod
    def multiply_polynomials(f, g, modulus):
        result_length = len(f) + len(g) - 1
        result = [0] * result_length
        for i in range(len(f)):
            for j in range(len(g)):
                result[i+j] = (result[i+j] + f[i] * g[j]) % modulus
        return result
    
    @staticmethod
    def ring_multiply_polynomials(f, g, modulus, n):
        result = MathOperations.multiply_polynomials(f, g, modulus)
        if len(result) > n:
            for i in range(n, len(result)):
                result[i%n] = (result[i%n] - result[i]) % modulus
            result = result[:n]
        return result
    
    @staticmethod
    def add_vectors(a, b, modulus, k):
        result = []
        for i in range(k):
            result.append(MathOperations.add_polynomials(a[i], b[i], modulus))
        return result
    
    @staticmethod
    def multiply_vectors(a, b, modulus, k, n):
        result = []
        for i in range(k):
            multiplied = MathOperations.ring_multiply_polynomials(a[i], b[i], modulus, n)
            result = MathOperations.add_polynomials(result, multiplied, modulus)
        return result

    @staticmethod
    def mod_s(r, q):
        r = r % q
        if q % 2 == 0:
            if r <= q // 2:
                result = r
            else:
                result = r - q
        else:
            if r <= (q - 1) // 2:
                result = r
            else:
                result = r - q
        return result

    @staticmethod
    def round_q(x, q):
        x_prime = MathOperations.mod_s(x, q)
        if -int(q/4) <= x_prime <= int(q/4):
            return 0
        else:
            return 1
        
    @staticmethod
    def round_polynomial(f, q):
        return [MathOperations.round_q(coefficient, q) for coefficient in f]

    @staticmethod
    def multiply_polynomial_by_scalar(f, scalar, modulus):
        return [(coefficient * scalar) % modulus for coefficient in f]

    @staticmethod
    def multiply_matrix_by_vector(A, b, modulus, k, n):
        result = [[] for _ in range(k)]

        for row_A in range(k):
            result[row_A] = MathOperations.multiply_vectors(A[row_A], b, modulus, k, n)

        return result

    @staticmethod
    def matrix_transpose(A, k, n):
        result = [[0] * k for _ in range(n)]
        for i in range(k):
            for j in range(n):
                result[j][i] = A[i][j]
        return result
    
    @staticmethod
    def bitstring_to_polynomial(m, n):
        f = [0] * n
        for i in range(n):
            if m[i] == "1":
                f[i] = 1
        return f

    @staticmethod
    def polynomial_to_bitstring(f, n):
        m = ""
        for i in range(n):
            if f[i] == 1:
                m += "1"
            else:
                m += "0"
        return m
    
    @staticmethod
    def text_in_bytes_to_bitstring(text_in_bytes, n):
        bitstring = ""
        
        for byte in text_in_bytes:
            binary_string = format(byte, '08b')
            bitstring += binary_string

        if len(bitstring) < n:
            bitstring += '0' * (n - len(bitstring))

        return bitstring
    
    @staticmethod
    def bitstring_to_text_in_bytes(bitstring, n):
        chunks_8bits = [bitstring[i:i+8] for i in range(0, n, 8)]
        bytes_list = bytes(int(byte, 2) for byte in chunks_8bits)
    
        return bytes_list

In [174]:
class CommunicationParty:
    def __init__(self, parameters: KyberParameters):
        self.parameters = parameters
        self.A = None
        self.s = None
        self.e = None
        self.t = None

    def generate_public_key(self):

        def select_A(q, k, n):
            A = []
            for i in range(k):
                row = []
                for j in range(k):
                    polynomial = [randint(0, q-1) for _ in range(n)]
                    row.append(polynomial)
                A.append(row)
            return A

        def select_s(k, n, eta1):
            s = []
            for i in range(k):
                polynomial = [randint(-eta1, eta1) for _ in range(n)]
                s.append(polynomial)
            return s

        def select_e(k, n, eta1):
            e = []
            for i in range(k):
                polynomial = [randint(-eta1, eta1) for _ in range(n)]
                e.append(polynomial)
            return e


        self.A = select_A(self.parameters.q, self.parameters.k, self.parameters.n)
        self.s = select_s(self.parameters.k, self.parameters.n, self.parameters.eta1)
        self.e = select_e(self.parameters.k, self.parameters.n, self.parameters.eta1)

        As = MathOperations.multiply_matrix_by_vector(self.A, self.s, self.parameters.q, self.parameters.k, self.parameters.n)
        self.t = MathOperations.add_vectors(As, self.e, self.parameters.q, self.parameters.k)

        return (self.A, self.t)

    def encrypt(self, text_in_bytes):
        def select_r(k, n, eta1):
            r = []
            for i in range(k):
                polynomial = [randint(-eta1, eta1) for _ in range(n)]
                r.append(polynomial)
            return r

        def select_e1(k, n, eta2):
            e1 = []
            for i in range(k):
                polynomial = [randint(-eta2, eta2) for _ in range(n)]
                e1.append(polynomial)
            return e1

        def select_e2(n, eta2):
            e2 = [randint(-eta2, eta2) for _ in range(n)]
            return e2

        bitstring = MathOperations.text_in_bytes_to_bitstring(text_in_bytes, self.parameters.n)
        m = MathOperations.bitstring_to_polynomial(bitstring, self.parameters.n)

        r = select_r(self.parameters.k, self.parameters.n, self.parameters.eta1)
        e1 = select_e1(self.parameters.k, self.parameters.n, self.parameters.eta2)
        e2 = select_e2(self.parameters.n, self.parameters.eta2)

        AT = MathOperations.matrix_transpose(self.A, self.parameters.k, self.parameters.k)
        ATr = MathOperations.multiply_matrix_by_vector(AT, r, self.parameters.q, self.parameters.k, self.parameters.n)
        u = MathOperations.add_vectors(ATr, e1, self.parameters.q, self.parameters.k)

        tTr = MathOperations.multiply_vectors(self.t, r, self.parameters.q, self.parameters.k, self.parameters.n)
        half_qm = MathOperations.multiply_polynomial_by_scalar(m, (self.parameters.q+1)//2, self.parameters.q)
        tTr_e2 = MathOperations.add_polynomials(tTr, e2, self.parameters.q)
        v = MathOperations.add_polynomials(tTr_e2, half_qm, self.parameters.q)
        
        return (u, v)

    def decrypt(self, ciphertext):
        u, v = ciphertext
        sTu = MathOperations.multiply_vectors(self.s, u, self.parameters.q, self.parameters.k, self.parameters.n)
        v_sub_sTu = MathOperations.subtract_polynomials(v, sTu, self.parameters.q)
        m_recovered = MathOperations.round_polynomial(v_sub_sTu, self.parameters.q)
        bitstring_recovered = MathOperations.polynomial_to_bitstring(m_recovered, self.parameters.n)
        text_in_bytes_recovered = MathOperations.bitstring_to_text_in_bytes(bitstring_recovered, self.parameters.n)

        return text_in_bytes_recovered

In [175]:
q = 3329
n = 256
k = 3
eta1 = 2
eta2 = 2

In [176]:
poem_en = '''Stopping by Woods on a Snowy Evening

He gives his harness bells a shake
To ask if there is some mistake.
The only other sound's the sweep
Of easy wind and downy flake.

The woods are lovely, dark and deep,
But I have promises to keep,
And miles to go before I sleep,
And miles to go before I sleep.'''

In [177]:
poem_pl = '''Stepy akermańskie

Wpłynąłem na suchego przestwór oceanu,
Wóz nurza się w zieloność i jak łódka brodzi,
Śród fali łąk szumiących, śród kwiatów powodzi,
Omijam koralowe ostrowy burzanu.

Już mrok zapada, nigdzie drogi ni kurhanu;
Patrzę w niebo, gwiazd szukam przewodniczek łodzi;
Tam z dala błyszczy obłok? tam jutrzenka wschodzi?
To błyszczy Dniestr, to weszła lampa Akermanu'''

In [181]:
kyber_parameters = KyberParameters(q, n, k, eta1, eta2)
Alice = CommunicationParty(kyber_parameters)
Bob = CommunicationParty(kyber_parameters)

Alice_public_key = Alice.generate_public_key()

Bob.A = Alice_public_key[0]
Bob.t = Alice_public_key[1]

for poem in [poem_en, poem_pl]:

    Alice_recovered_bytes = b""
    poem_bytes = poem.encode('utf-8')
    for i in range(0, len(poem_bytes), kyber_parameters.n // 8):
        chunk_of_bytes = poem_bytes[i:i + kyber_parameters.n // 8]
        Bob_ciphertext = Bob.encrypt(chunk_of_bytes) 
        Alice_recovered_bytes += Alice.decrypt(Bob_ciphertext)
    
    Alice_plaintext_recovered = Alice_recovered_bytes.decode('utf-8', errors='ignore').rstrip('\x00')

    assert Alice_plaintext_recovered == poem, "Recovered plaintext is NOT the same as Original plaintext!"

    print(Alice_plaintext_recovered)
    print("\n\n")

Stopping by Woods on a Snowy Evening

He gives his harness bells a shake
To ask if there is some mistake.
The only other sound's the sweep
Of easy wind and downy flake.

The woods are lovely, dark and deep,
But I have promises to keep,
And miles to go before I sleep,
And miles to go before I sleep.



Stepy akermańskie

Wpłynąłem na suchego przestwór oceanu,
Wóz nurza się w zieloność i jak łódka brodzi,
Śród fali łąk szumiących, śród kwiatów powodzi,
Omijam koralowe ostrowy burzanu.

Już mrok zapada, nigdzie drogi ni kurhanu;
Patrzę w niebo, gwiazd szukam przewodniczek łodzi;
Tam z dala błyszczy obłok? tam jutrzenka wschodzi?
To błyszczy Dniestr, to weszła lampa Akermanu





## V2b - optimizations

In [ ]:
# use 256bit ro to generate A, then public key is (ro, t) instead of (A, t) TODO

In [ ]:
def sampleNTT(ro, j, i):
    
    pass

def generate_A_from_ro(ro, q, k, n):
    A = []
    for i in range(k):
        row = []
        for j in range(k):
            polynomial = [sampleNTT(ro,j,i) for _ in range(n)]
            row.append(polynomial)
        A.append(row)
    return A

In [ ]:
def compress_integer(x, q, d):
    y = (((2**d) * x) + (q//2)) // q
    return y%(2**d)

def decompress_integer(y, q, d):
    x = ((q * y) + (2**d // 2)) // (2**d)
    return x%q

In [6]:
q = 19
d = 2

for x in range(0, 19):
    y = compress_integer(x, q, d)
    x_prime = decompress_integer(y, q, d)

    print(f"x: {x}, compressed: {y}, decompressed: {x_prime}")

x: 0, compressed: 0, decompressed: 0
x: 1, compressed: 0, decompressed: 0
x: 2, compressed: 0, decompressed: 0
x: 3, compressed: 1, decompressed: 5
x: 4, compressed: 1, decompressed: 5
x: 5, compressed: 1, decompressed: 5
x: 6, compressed: 1, decompressed: 5
x: 7, compressed: 1, decompressed: 5
x: 8, compressed: 2, decompressed: 10
x: 9, compressed: 2, decompressed: 10
x: 10, compressed: 2, decompressed: 10
x: 11, compressed: 2, decompressed: 10
x: 12, compressed: 3, decompressed: 14
x: 13, compressed: 3, decompressed: 14
x: 14, compressed: 3, decompressed: 14
x: 15, compressed: 3, decompressed: 14
x: 16, compressed: 3, decompressed: 14
x: 17, compressed: 0, decompressed: 0
x: 18, compressed: 0, decompressed: 0


In [15]:
def compress_polynomial(f, q, d):
    return [compress_integer(coefficient, q, d) for coefficient in f]

def decompress_polynomial(f, q, d):
    return [decompress_integer(coefficient, q, d) for coefficient in f]

def calculate_error_polynomial(f, g, q):
    polynomial = subtract_polynomials(f, g, q)
    polynomial_mod_s = [mod_s(coefficient, q) for coefficient in polynomial]
    return polynomial_mod_s

In [19]:
q = 3329
d = 10

print(f"error should be at most {(q + ((2**(d+1))//2)) // (2**(d+1))}")

f = [223,1438,3280,798]
f_compressed = compress_polynomial(f, q, d)
f_decompressed = decompress_polynomial(f_compressed, q, d)
error_polynomial = calculate_error_polynomial(f, f_decompressed, q)

print(f_compressed)
print(f_decompressed)
print(error_polynomial)

error should be at most 2
[69, 442, 1009, 245]
[224, 1437, 3280, 796]
[-1, 1, 0, 2]


In [21]:
q = 3329
d = 4

print(f"error should be at most {(q + ((2**(d+1))//2)) // (2**(d+1))}")

f = [223,1438,3280,798]
f_compressed = compress_polynomial(f, q, d)
f_decompressed = decompress_polynomial(f_compressed, q, d)
error_polynomial = calculate_error_polynomial(f, f_decompressed, q)

print(f_compressed)
print(f_decompressed)
print(error_polynomial)

error should be at most 104
[1, 7, 0, 4]
[208, 1456, 0, 832]
[15, -18, -49, -34]


In [ ]:
# Ciphertext compression: (u, v) -> (c1, c2) 
# where c1 = compress(u, du) and c2 = compress(v, dv)
# ML-KEM-768 parameters: du = 10, dv = 4

In [22]:
# Fast Polynomial Multiplication -> use NTT TODO

## V2c - full scheme

- ML-KEM-768 domain parameters:
    | parameter | value |
    |-----------|-------|
    |     q     | 3329  |
    |     n     | 256   |
    |     k     | 3     |
    |     eta1  | 2     |
    |     eta2  | 2     |
    |     du    | 10    |
    |     dv    | 4     |
    

- Kyber-PKE key generation - Alice does:
    1. select 256bit ro, compute k by k matrix A = expand(ro) 
    2. select vectors s and e - each contains k polynomials with n coefficients from -eta1 to +eta
    3. compute t = As + e
    4. encryption (public) key is (ro, t) and decryption (private) key is s

- Kyber-PKE encryption - to encrypt message (bitstring) m, Bob does:
    1. obtain a copy of Alice's public key (ro, t) and compute A = expand(ro)
    2. select vectors r and e1 - k polynomials with n coefficients (r uses eta1, e1 uses eta2) and polynomial e2 with n coefficients using eta2
    3. compute u = ATr + e1 and v = tTr + e2 + m*((q+1)//2)
    4. compute c1 = compress(u, du) and c2 = compress(v, dv)
    5. ciphertext is c = (c1, c2)

- Kyber-PKE decryption - to decrypt c=(c1, c2), Alice does:
    1. compute u' = decompress(c1, du) and v' = decompress(c2, dv)
    2. compute m = round_q(v' - sTu')

## V2d - Kyber KEM

- ML-KEM-768 domain parameters:

    | parameter | value |
    |-----------|-------|
    |     q     | 3329  |
    |     n     | 256   |
    |     k     | 3     |
    |     eta1  | 2     |
    |     eta2  | 2     |
    |     du    | 10    |
    |     dv    | 4     |
-----------------------------

- Fujisaki-Okamoto Transform:
    - H(s) = SHA3-256(s)
    - J(s) = SHAKE256(s, 8*32)
    - G(c) = SHA3-512(c)


- Kyber-KEM key generation - Alice does:
    1. use Kyber-PKE key generation to select Kyber-PKE encryption key (ro, t) and decryption key s
    2. select 256bit z
    3. encapsulation key is ek = (ro, t) and decapsulation key is dk = (s, ek, H(ek), z)


- Kyber-KEM encapsulation - to establish a shared secret key with Alice, Bob does:
    1. obtain a copy of Alice's encapsulation key ek
    2. select 256bit m
    3. compute h = H(ek) and (K, R) = G(m,h) where K,R are 256bit
    4. use Kyber-PKE encryption to encrypt m with key ek, using R to generate random quantities needed, then call the resulting ciphertext c
    5. output the shared secret key K and ciphertext c 


- Kyber-KEM decapsulation - to recover secret key K from c using dk=(s,ek,H(ek),z), Alice does:
    1. use Kyber-PKE decryption to decrypt c with key s, call the resulting plaintext m'
    2. compute (K', R') = G(m', H(ek))
    3. compute K_bar = J(z, c)
    4. use Kyber-PKE encryption to encrypt m' with key ek, using R' to generate random quantities needed, then call the resulting ciphertext c'
    5. if c is NOT equal c' then return K_bar
    6. if c is equal c' then return K'

In [23]:
#TODO

# V3

## V3a - Dilithium (Toy version)

## V3b - Dilithium scheme (without t compression)

## V3c - Dilithium t compression

## V3d - Dilithium full scheme

# V4

## V4a - Dilithium NTT

## V4b - Kyber NTT